> **Versión corregida (v2.13, línea v1 de despliegue, ejecución verificada)** — generada a partir de `notebooks/11_visualizaciones_conclusiones_finales.ipynb` sin sobrescribirlo.
>
> Cambios aplicados:
> - Celda de inicialización: se sustituyen la detección/montaje de Drive y el clonado de `main` por el bootstrap portable (`PROJECT_ROOT`/`REPO_ROOT`, rama `feature/entorno-capacidad-v2`).
> - Renombrado semántico: el antiguo `PROJECT_ROOT` (copia del repositorio) pasa a `REPO_ROOT`; `PERSISTENT_ROOT` se conserva como raíz de la línea v1 (= `PROJECT_ROOT`, o `PROJECT_ROOT/prueba_linea_v1` en modo prueba).
>
> Rutas: `PROJECT_ROOT` = datos/modelos/resultados (Drive en Colab, o `PIDA_PROJECT_ROOT`); `REPO_ROOT` = código. Modo prueba: `PIDA_MODO_PRUEBA=1` (cohorte pequeña, pocos pasos, salida en `prueba_linea_v1/`).


# PIDA-RL Diabetes — 11. Visualizaciones y conclusiones finales

Este notebook transforma los resultados de los notebooks 09 y 10 en tablas de decisión, visualizaciones finales, una conclusión técnica reproducible y un conjunto compacto de insumos para el dashboard Streamlit.

> **Alcance:** todos los hallazgos corresponden a un entorno de simulación con datos públicos y supuestos explícitos. No demuestran efectividad clínica, causalidad, seguridad real ni justifican un despliegue institucional autónomo.


## 1. Objetivo

El objetivo es consolidar evidencia de rendimiento, capacidad, seguridad simulada y priorización por nivel de riesgo. La salida incluye:

- Una tabla final de decisión para las seis políticas.
- Figuras comparativas para documentación y dashboard.
- Una recomendación técnica condicionada a criterios verificables.
- Archivos CSV compactos que Streamlit pueda cargar sin leer trazas de gran tamaño.


## 2. Inicialización


In [ ]:
# === Bootstrap portable: PROJECT_ROOT (corrección v2.12) ===
# PROJECT_ROOT = raíz persistente de datos, modelos y resultados.
# REPO_ROOT    = copia local del código (repositorio GitHub).
# Prioridad: variables de entorno PIDA_PROJECT_ROOT / PIDA_REPO_ROOT,
# después Google Colab + Drive y, por último, ejecución local.
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import importlib
import importlib.util
import subprocess

PROJECT_NAME = 'pida-rl-datos-publicos'

REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
BRANCH = os.environ.get('PIDA_BRANCH', 'main')
PROJECT_FOLDER_NAME = 'PIDA-RL-Diabetes'
IN_COLAB = 'google.colab' in sys.modules

DRIVE_ROOT = None
if os.environ.get('PIDA_PROJECT_ROOT'):
    PROJECT_ROOT = Path(os.environ['PIDA_PROJECT_ROOT']).expanduser().resolve()
elif IN_COLAB:
    DRIVE_ROOT = next(
        (p for p in [Path('/content/drive'), Path('/content/gdrive')] if (p / 'MyDrive').is_dir()),
        None,
    )
    if DRIVE_ROOT is None:
        from google.colab import drive
        mount_point = Path('/content/drive')
        if mount_point.exists() and any(mount_point.iterdir()):
            mount_point = Path('/content/gdrive')
        drive.mount(str(mount_point), force_remount=False)
        DRIVE_ROOT = mount_point
    PROJECT_ROOT = DRIVE_ROOT / 'MyDrive' / PROJECT_FOLDER_NAME
else:
    PROJECT_ROOT = (Path.cwd() / PROJECT_FOLDER_NAME).resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


def _buscar_repo(inicio):
    for candidato in [inicio, *inicio.parents]:
        if (candidato / 'src' / 'environment' / 'diabetes_followup_env.py').is_file():
            return candidato
    return None


if os.environ.get('PIDA_REPO_ROOT'):
    REPO_ROOT = Path(os.environ['PIDA_REPO_ROOT']).expanduser().resolve()
else:
    REPO_ROOT = _buscar_repo(Path.cwd().resolve()) or (
        Path('/content/pida-rl-datos-publicos') if IN_COLAB
        else Path.cwd().resolve() / 'pida-rl-datos-publicos'
    )
if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])
elif not (REPO_ROOT / 'src').is_dir():
    raise RuntimeError(f'REPO_ROOT no contiene la carpeta src: {REPO_ROOT}')
elif IN_COLAB and (REPO_ROOT / '.git').is_dir():
    # En Colab se alinea la copia temporal con la rama de trabajo (sin push ni commit).
    for args in (['fetch', 'origin', BRANCH], ['checkout', BRANCH], ['pull', '--ff-only', 'origin', BRANCH]):
        subprocess.run(['git', '-C', str(REPO_ROOT), *args], check=False)

SRC_ROOT = REPO_ROOT / 'src'
for ruta in (str(REPO_ROOT), str(SRC_ROOT)):
    if ruta not in sys.path:
        sys.path.insert(0, ruta)
os.chdir(REPO_ROOT)

# Estructura persistente común (antes: rutas fijas /content/drive/MyDrive/PIDA-RL-Diabetes).
PERSISTENT_ROOT = PROJECT_ROOT  # alias de compatibilidad con los notebooks originales
MYDRIVE_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_ROOT / 'raw'
INTERIM_DATA_DIR = DATA_ROOT / 'interim'
PROCESSED_DATA_DIR = DATA_ROOT / 'processed'
METADATA_DIR = DATA_ROOT / 'metadata'
SPECIFICATIONS_DIR = DATA_ROOT / 'specifications'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'

# Modo prueba: cohorte pequeña y pocos episodios (PIDA_MODO_PRUEBA=1).
MODO_PRUEBA = os.environ.get('PIDA_MODO_PRUEBA', '0') == '1'
N_COHORTE_PRUEBA = int(os.environ.get('PIDA_N_COHORTE', '120'))
SEMILLA_MUESTRA_PRUEBA = 2026
RESULTS_V2_DIR = RESULTS_DIR / ('capacidad_v2_prueba' if MODO_PRUEBA else 'capacidad_v2')


def muestra_cohorte_prueba(cohorte, n=None, semilla=SEMILLA_MUESTRA_PRUEBA):
    """Devuelve (cohorte, factor). En modo prueba toma una submuestra
    estratificada por categoria_riesgo (bajo/medio/alto); no crea columnas."""
    n = N_COHORTE_PRUEBA if n is None else int(n)
    if not MODO_PRUEBA or len(cohorte) <= n:
        return cohorte, 1.0
    fraccion = n / len(cohorte)
    muestra = (
        cohorte.groupby('categoria_riesgo', group_keys=False)
        .sample(frac=fraccion, random_state=semilla)
        .sort_values('FOLIO_INT')
        .reset_index(drop=True)
    )
    return muestra, len(muestra) / len(cohorte)


def escalar_capacidad(capacidad, factor):
    """Mantiene la presión de capacidad (recursos/persona) al reducir la cohorte."""
    return capacidad if factor >= 1.0 else max(1, int(round(capacidad * factor)))

print('Entorno:', 'Google Colab' if IN_COLAB else 'local')
print('PROJECT_ROOT (datos/resultados):', PROJECT_ROOT)
print('REPO_ROOT (código):', REPO_ROOT)
print('Modo prueba:', MODO_PRUEBA)

# --- Línea v1 de despliegue (entorno autocontenido) ---
# En modo prueba se escribe en una carpeta aparte para no sobrescribir resultados completos ni el dashboard publicado.
LINEA_V1_ROOT = PROJECT_ROOT / 'prueba_linea_v1' if MODO_PRUEBA else PROJECT_ROOT
PERSISTENT_ROOT = LINEA_V1_ROOT
COHORTE_CANONICA = PROJECT_ROOT / 'data' / 'processed' / 'cohorte_diabetes.csv'
print('Raíz de la línea v1 (PERSISTENT_ROOT):', PERSISTENT_ROOT)

# --- Celda original (sin montaje de Drive ni clonado fijo) ---
from pathlib import Path
import os
import subprocess
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TABLES_DIR = PERSISTENT_ROOT / 'results' / 'tables'
FIGURES_DIR = PERSISTENT_ROOT / 'results' / 'figures'
METADATA_DIR = PERSISTENT_ROOT / 'data' / 'metadata'
DASHBOARD_DATA_DIR = PERSISTENT_ROOT / 'dashboard_data'

for directory in [TABLES_DIR, FIGURES_DIR, METADATA_DIR, DASHBOARD_DATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_ROOT)
print('Proyecto temporal:', REPO_ROOT)
print('Tablas:', TABLES_DIR)
print('Figuras:', FIGURES_DIR)
print('Datos para dashboard:', DASHBOARD_DATA_DIR)


## 3. Archivos de entrada

Se cargan solo resultados resumidos de los notebooks 09 y 10. Las trazas grandes no se utilizan para las visualizaciones finales ni para el dashboard.


In [ ]:
INPUT_FILES = {
    'ranking_tecnico': TABLES_DIR / 'ranking_tecnico_modelos_v1.csv',
    'resumen_modelos': TABLES_DIR / 'resumen_modelos_v1.csv',
    'cobertura_riesgo_n09': TABLES_DIR / 'cobertura_por_riesgo_modelos_v1.csv',
    'equidad_riesgo': TABLES_DIR / 'equidad_por_riesgo_modelos_v1.csv',
    'brechas_riesgo': TABLES_DIR / 'brechas_por_riesgo_modelos_v1.csv',
    'alto_riesgo_semilla': TABLES_DIR / 'alto_riesgo_por_semilla_modelos_v1.csv',
    'resumen_equidad_riesgo': TABLES_DIR / 'resumen_equidad_riesgo_modelos_v1.csv',
}

for name, path in INPUT_FILES.items():
    print(f'{name:28s} | existe: {path.is_file()} | {path.name}')

missing_files = [str(path) for path in INPUT_FILES.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        'Faltan insumos de los notebooks 09 o 10:\n' + '\n'.join(missing_files)
    )


## 4. Carga y validación


In [ ]:
data = {name: pd.read_csv(path) for name, path in INPUT_FILES.items()}

df_ranking = data['ranking_tecnico'].copy()
df_resumen = data['resumen_modelos'].copy()
df_cobertura_riesgo = data['cobertura_riesgo_n09'].copy()
df_equidad_riesgo = data['equidad_riesgo'].copy()
df_brechas_riesgo = data['brechas_riesgo'].copy()
df_alto_riesgo_semilla = data['alto_riesgo_semilla'].copy()
df_resumen_equidad_riesgo = data['resumen_equidad_riesgo'].copy()

expected_policies = {
    'aleatoria',
    'reglas_riesgo',
    'reglas_capacidad',
    'dqn_v1',
    'q_learning_v1',
    'ppo_v1',
}

for name, frame in data.items():
    if 'politica' not in frame.columns:
        raise KeyError(f'{name}: falta la columna politica. Columnas disponibles: {list(frame.columns)}')
    found_policies = set(frame['politica'].astype(str).unique())
    missing_policies = expected_policies - found_policies
    if missing_policies:
        raise ValueError(f'{name}: faltan políticas: {sorted(missing_policies)}')

required_ranking_columns = [
    'ranking_tecnico',
    'politica',
    'puntaje_tecnico',
    'recompensa_media',
    'cobertura_alto_riesgo_media',
    'eventos_adversos_medios',
    'recursos_usados_medios',
    'excesos_capacidad_medios',
    'respeta_capacidad_en_promedio',
    'clasificacion',
]
missing_ranking_columns = [column for column in required_ranking_columns if column not in df_ranking.columns]
if missing_ranking_columns:
    raise KeyError(f'ranking_tecnico: faltan columnas: {missing_ranking_columns}')

required_equity_columns = [
    'politica',
    'bajo',
    'medio',
    'alto',
    'brecha_absoluta_riesgo_pp',
    'brecha_alto_menos_bajo_pp',
]
missing_equity_columns = [column for column in required_equity_columns if column not in df_resumen_equidad_riesgo.columns]
if missing_equity_columns:
    raise KeyError(f'resumen_equidad_riesgo: faltan columnas: {missing_equity_columns}')

print('Validación de insumos completada.')
display(df_ranking.sort_values('ranking_tecnico'))


## 5. Tabla final de decisión

La tabla integra rendimiento, operación y priorización de riesgo. El campo `candidata_operativa` no es una aprobación clínica: identifica una política de referencia que cumple criterios técnicos dentro del simulador.


In [ ]:
equity_columns = [
    'politica',
    'bajo',
    'medio',
    'alto',
    'brecha_absoluta_riesgo_pp',
    'brecha_alto_menos_bajo_pp',
    'alerta_cobertura_minima_riesgo',
    'alerta_exceso_capacidad',
]
available_equity_columns = [column for column in equity_columns if column in df_resumen_equidad_riesgo.columns]

df_decision_final = (
    df_ranking
    .merge(
        df_resumen_equidad_riesgo[available_equity_columns],
        on='politica',
        how='left',
        suffixes=('', '_equidad'),
    )
)

if df_decision_final[['bajo', 'medio', 'alto']].isna().any().any():
    raise ValueError('La tabla final tiene cobertura por riesgo faltante después de integrar los resultados.')

df_decision_final['prioriza_alto_riesgo'] = df_decision_final['brecha_alto_menos_bajo_pp'] > 0
df_decision_final['cubre_alto_riesgo'] = df_decision_final['alto'] > 0
df_decision_final['sin_alerta_cobertura_minima'] = ~df_decision_final['alerta_cobertura_minima_riesgo'].fillna(True).astype(bool)
df_decision_final['candidata_operativa'] = (
    (df_decision_final['ranking_tecnico'] == 1)
    & df_decision_final['respeta_capacidad_en_promedio'].astype(bool)
    & df_decision_final['cubre_alto_riesgo']
)

df_decision_final = df_decision_final.sort_values(['ranking_tecnico', 'politica']).reset_index(drop=True)

decision_columns = [
    'ranking_tecnico',
    'politica',
    'clasificacion',
    'puntaje_tecnico',
    'recompensa_media',
    'cobertura_alto_riesgo_media',
    'bajo',
    'medio',
    'alto',
    'brecha_alto_menos_bajo_pp',
    'eventos_adversos_medios',
    'recursos_usados_medios',
    'excesos_capacidad_medios',
    'respeta_capacidad_en_promedio',
    'candidata_operativa',
]
display(df_decision_final[decision_columns].round(3))


## 6. Conclusión reproducible

La recomendación se genera a partir de la tabla anterior. Se evita declarar superioridad clínica: el resultado identifica la política con mejor desempeño técnico bajo las reglas y supuestos del simulador.


In [ ]:
candidate_rows = df_decision_final.loc[df_decision_final['candidata_operativa']].copy()

if candidate_rows.empty:
    selected_policy = df_decision_final.iloc[0]['politica']
    recommendation_status = 'sin_candidata_operativa_automatica'
    recommendation_text = (
        f'La política mejor posicionada en el ranking técnico es {selected_policy}, '
        'pero no cumple todos los criterios operativos definidos para marcar una candidata automática. '
        'Se requiere revisión humana de capacidad, cobertura y supuestos del simulador.'
    )
else:
    selected_policy = candidate_rows.iloc[0]['politica']
    recommendation_status = 'candidata_operativa_en_simulacion'
    recommendation_text = (
        f'Bajo el protocolo de evaluación del simulador, {selected_policy} es la candidata operativa de referencia: '
        'ocupa el primer lugar técnico, mantiene cobertura de alto riesgo y respeta la capacidad promedio. '
        'Esta conclusión no equivale a validación clínica ni a autorización de despliegue.'
    )

selected_metrics = df_decision_final.loc[df_decision_final['politica'] == selected_policy].iloc[0]

conclusion_records = [
    {
        'tipo': 'resultado_principal',
        'contenido': recommendation_text,
    },
    {
        'tipo': 'rendimiento',
        'contenido': (
            f'{selected_policy}: puntaje técnico={selected_metrics["puntaje_tecnico"]:.3f}; '
            f'recompensa media={selected_metrics["recompensa_media"]:.3f}; '
            f'cobertura de alto riesgo={selected_metrics["alto"]:.3f}%.'
        ),
    },
    {
        'tipo': 'capacidad',
        'contenido': (
            f'{selected_policy}: excesos medios de capacidad={selected_metrics["excesos_capacidad_medios"]:.3f}; '
            f'respeta capacidad en promedio={bool(selected_metrics["respeta_capacidad_en_promedio"])}.'
        ),
    },
    {
        'tipo': 'equidad_por_riesgo',
        'contenido': (
            f'{selected_policy}: cobertura bajo/medio/alto='
            f'{selected_metrics["bajo"]:.3f}%/{selected_metrics["medio"]:.3f}%/{selected_metrics["alto"]:.3f}%; '
            f'brecha alto menos bajo={selected_metrics["brecha_alto_menos_bajo_pp"]:.3f} puntos porcentuales.'
        ),
    },
    {
        'tipo': 'limitacion',
        'contenido': (
            'La auditoría disponible es por nivel de riesgo. No se calcularon resultados por sexo o edad porque las trazas de evaluación agregadas no contienen atributos demográficos ni identificador individual.'
        ),
    },
    {
        'tipo': 'siguiente_validacion',
        'contenido': (
            'Antes de uso institucional se requieren datos autorizados, validación clínica prospectiva, evaluación demográfica por subgrupos, revisión de gobernanza, trazabilidad y supervisión humana.'
        ),
    },
]

df_conclusiones_finales = pd.DataFrame(conclusion_records)
print('Estado de recomendación:', recommendation_status)
display(df_conclusiones_finales)


## 7. Visualizaciones finales


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
policy_order = df_decision_final['politica'].tolist()

fig, axes = plt.subplots(2, 2, figsize=(16, 11), constrained_layout=True)

rank_plot = df_decision_final.set_index('politica').reindex(policy_order)
rank_plot['puntaje_tecnico'].plot(kind='bar', ax=axes[0, 0], color='#4C78A8')
axes[0, 0].set_title('Puntaje técnico integrado')
axes[0, 0].set_xlabel('Política')
axes[0, 0].set_ylabel('Puntaje normalizado')
axes[0, 0].tick_params(axis='x', rotation=45)

coverage_plot = df_decision_final.set_index('politica')[['bajo', 'medio', 'alto']].reindex(policy_order)
coverage_plot.plot(kind='bar', ax=axes[0, 1], color=['#72B7B2', '#F2CF5B', '#E45756'])
axes[0, 1].set_title('Cobertura por nivel de riesgo')
axes[0, 1].set_xlabel('Política')
axes[0, 1].set_ylabel('Cobertura media (%)')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].legend(title='Riesgo')

capacity_plot = df_decision_final.set_index('politica').reindex(policy_order)
capacity_plot['excesos_capacidad_medios'].plot(kind='bar', ax=axes[1, 0], color='#E45756')
axes[1, 0].set_title('Excesos promedio de capacidad')
axes[1, 0].set_xlabel('Política')
axes[1, 0].set_ylabel('Excesos por episodio')
axes[1, 0].tick_params(axis='x', rotation=45)

stability_plot = (
    df_alto_riesgo_semilla
    .pivot(index='politica', columns='semilla', values='cobertura_porcentaje')
    .reindex(policy_order)
)
stability_plot.plot(kind='bar', ax=axes[1, 1])
axes[1, 1].set_title('Cobertura de alto riesgo por semilla')
axes[1, 1].set_xlabel('Política')
axes[1, 1].set_ylabel('Cobertura (%)')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].legend(title='Semilla')

final_figure_path = FIGURES_DIR / 'comparacion_final_politicas_v1.png'
fig.savefig(final_figure_path, dpi=200, bbox_inches='tight')
plt.show()
print('Figura final guardada:', final_figure_path)


## 8. Insumos para Streamlit

Se exportan archivos pequeños y listos para consumo. El dashboard debe cargar estos archivos resumidos en lugar de las trazas de más de 30 MB.


In [ ]:
DASHBOARD_DECISION_PATH = DASHBOARD_DATA_DIR / 'tabla_decision_final_v1.csv'
DASHBOARD_RISK_PATH = DASHBOARD_DATA_DIR / 'cobertura_por_riesgo_dashboard_v1.csv'
DASHBOARD_STABILITY_PATH = DASHBOARD_DATA_DIR / 'alto_riesgo_por_semilla_dashboard_v1.csv'
DASHBOARD_CONCLUSIONS_PATH = DASHBOARD_DATA_DIR / 'conclusiones_dashboard_v1.csv'
DASHBOARD_CONFIG_PATH = DASHBOARD_DATA_DIR / 'configuracion_dashboard_v1.json'

dashboard_decision = df_decision_final[decision_columns].copy()
dashboard_risk = df_equidad_riesgo.copy()
dashboard_stability = df_alto_riesgo_semilla.copy()

dashboard_decision.to_csv(DASHBOARD_DECISION_PATH, index=False, encoding='utf-8-sig')
dashboard_risk.to_csv(DASHBOARD_RISK_PATH, index=False, encoding='utf-8-sig')
dashboard_stability.to_csv(DASHBOARD_STABILITY_PATH, index=False, encoding='utf-8-sig')
df_conclusiones_finales.to_csv(DASHBOARD_CONCLUSIONS_PATH, index=False, encoding='utf-8-sig')

dashboard_config = {
    'version': '1.0',
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'selected_policy': selected_policy,
    'recommendation_status': recommendation_status,
    'policies': policy_order,
    'scope_note': 'Resultados de simulación con datos públicos y supuestos explícitos. No usar para decisiones clínicas reales.',
    'data_files': {
        'decision_table': DASHBOARD_DECISION_PATH.name,
        'risk_coverage': DASHBOARD_RISK_PATH.name,
        'high_risk_stability': DASHBOARD_STABILITY_PATH.name,
        'conclusions': DASHBOARD_CONCLUSIONS_PATH.name,
    },
}

with DASHBOARD_CONFIG_PATH.open('w', encoding='utf-8') as file:
    json.dump(dashboard_config, file, ensure_ascii=False, indent=2)

print('Insumos del dashboard guardados.')


## 9. Guardar resultados finales


In [ ]:
DECISION_PATH = TABLES_DIR / 'tabla_decision_final_modelos_v1.csv'
CONCLUSIONS_PATH = TABLES_DIR / 'conclusiones_finales_pida_v1.csv'
METADATA_PATH = METADATA_DIR / 'protocolo_visualizaciones_conclusiones_v1.json'

df_decision_final.to_csv(DECISION_PATH, index=False, encoding='utf-8-sig')
df_conclusiones_finales.to_csv(CONCLUSIONS_PATH, index=False, encoding='utf-8-sig')

final_metadata = {
    'version': '1.0',
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'selected_policy': selected_policy,
    'recommendation_status': recommendation_status,
    'selection_rule': 'ranking_tecnico == 1, respeta_capacidad_en_promedio == True y cobertura de alto riesgo > 0',
    'input_files': {name: str(path) for name, path in INPUT_FILES.items()},
    'outputs': {
        'decision_table': str(DECISION_PATH),
        'conclusions': str(CONCLUSIONS_PATH),
        'figure': str(final_figure_path),
        'dashboard_data_dir': str(DASHBOARD_DATA_DIR),
    },
    'limitations': [
        'Resultados basados en un simulador, no en seguimiento institucional real.',
        'La auditoría disponible es por nivel de riesgo; no hay trazas demográficas individuales para sexo o edad.',
        'Se requiere validación clínica, gobernanza y supervisión humana antes de cualquier uso real.',
    ],
}

with METADATA_PATH.open('w', encoding='utf-8') as file:
    json.dump(final_metadata, file, ensure_ascii=False, indent=2)

output_paths = [
    DECISION_PATH,
    CONCLUSIONS_PATH,
    final_figure_path,
    DASHBOARD_DECISION_PATH,
    DASHBOARD_RISK_PATH,
    DASHBOARD_STABILITY_PATH,
    DASHBOARD_CONCLUSIONS_PATH,
    DASHBOARD_CONFIG_PATH,
    METADATA_PATH,
]

for path in output_paths:
    size = path.stat().st_size if path.is_file() else 0
    print(path.name, '| existe:', path.is_file(), '| bytes:', size)

print('Visualizaciones, conclusión e insumos del dashboard completados y guardados.')


## Resultado esperado

El notebook debe finalizar con:

```text
Visualizaciones, conclusión e insumos del dashboard completados y guardados.
```

Los siguientes insumos permitirán construir el dashboard Streamlit: `tabla_decision_final_v1.csv`, `cobertura_por_riesgo_dashboard_v1.csv`, `alto_riesgo_por_semilla_dashboard_v1.csv`, `conclusiones_dashboard_v1.csv` y `configuracion_dashboard_v1.json`.
